In [ ]:
# Colab の場合はまずランタイムを GPU に変更してから実行
pip install --quiet diffusers==0.27.2 transformers accelerate safetensors

In [ ]:
# -----------------------------------------
# Original Color Palette
# -----------------------------------------
color_palette = {
    "white":        "#FFFFFF",
    "light_gray":   "#D3D3D3",
    "gray":         "#808080",
    "black":        "#000000",
    "green":        "#008000",
    "blue":         "#0000FF",
    "light_blue":   "#ADD8E6",
    "light_light_blue": "#E0FFFF",
    "yellow":       "#FFFF00",
    "orange":       "#FFA500",
    "dark_orange":  "#FF8C00",
    "purple":       "#800080",
}

# =============================================================================
# Stable Diffusion VAE ― Unconditional Generation Latent ➜ Decoded Image
# =============================================================================
import torch, random, numpy as np
import matplotlib.pyplot as plt
from diffusers import StableDiffusionPipeline

# ---------------------------------------------------------------------
# 0. 再現性の完全固定
# ---------------------------------------------------------------------
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ---------------------------------------------------------------------
# 1. Pipeline 読み込み（fp16, GPU）
#    * VAE・UNet など一式を読み込みます（≈4 GB）。必要なら別 VAE のみに変更可
# ---------------------------------------------------------------------
model_id = "runwayml/stable-diffusion-v1-5"
pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
    variant="fp16"
).to("cuda")
pipe.set_progress_bar_config(disable=True)  # Colab のログを簡素化

# ---------------------------------------------------------------------
# 2. 「プロンプトなし」で 1 枚生成し，最終 latent を取得
#    output_type="latent" にすると decode をスキップし，(4, 64, 64) Tensor が得られる
# ---------------------------------------------------------------------
with torch.autocast("cuda"):
    gen = torch.Generator(device="cuda").manual_seed(SEED)
    latent_out = pipe(
        prompt       = "",               # unconditional
        num_inference_steps = 50,
        output_type  = "latent",
        generator    = gen,
    ).images[0]        # Tensor, shape [4, 64, 64], dtype=float16, device=cuda

# ---------------------------------------------------------------------
# 3. VAE デコーダへ入力し，RGB 画像 (3, 512, 512) を取得
#    * Stable Diffusion では latent / scaling_factor (=0.18215) が標準
# ---------------------------------------------------------------------
scaling_factor = pipe.vae.config.scaling_factor  # 0.18215
with torch.no_grad():
    decoded = pipe.vae.decode(latent_out.unsqueeze(0) / scaling_factor).sample
decoded = (decoded.clamp(-1, 1) + 1) / 2  # [-1,1] ➜ [0,1]
decoded = decoded.squeeze(0).cpu().float()

# ---------------------------------------------------------------------
# 4. 可視化（subplot を避け，各 Figure は単独）
# ---------------------------------------------------------------------
# 4-A) latent tensor の各チャネル（Grayscale）
for ch in range(4):
    plt.figure(figsize=(4, 4))
    img = latent_out[ch].cpu().float()
    img_min, img_max = img.min(), img.max()
    # 0-1 へ正規化して可視化
    img_norm = (img - img_min) / (img_max - img_min + 1e-8)
    plt.imshow(img_norm, cmap="gray")
    plt.title(f"Latent Channel {ch} (normed)")
    plt.axis("off")
    plt.show()

# 4-B) デコーダ出力画像
plt.figure(figsize=(6, 6))
plt.imshow(decoded.permute(1, 2, 0).numpy())
plt.title("Decoded Image from VAE (unconditional)")
plt.axis("off")
plt.show()